# Цифровая обработка сигналов - Лабораторная работа 7
# Квадратурная фазовая манипуляция (QPSK)

## Введение

В этой работе моделируется полная цифровая система связи с модуляцией **QPSK** (англ. *Quadrature Phase Shift Keying* - квадратурная фазовая манипуляция, в отечественной литературе также ФМ-4): от генерации битов на передатчике до восстановления битов на приемнике и оценки качества связи. Работа объединяет темы предыдущих лабораторных: дискретизацию, спектральный анализ, КИХ-фильтрацию и свертку.

**Цифровая модуляция** - это отображение потока битов на параметры гармонического колебания (несущей), которое можно передать по физическому каналу (радиоэфир, кабель, оптоволокно). Несущая $A\cos(\omega_0 t + \varphi)$ имеет три параметра, поэтому различают три базовых вида манипуляции:

| Вид | Что меняется | Пример |
| --- | --- | --- |
| ASK (амплитудная манипуляция, АМн) | амплитуда $A$ | оптические линии, RFID |
| FSK (частотная манипуляция, ЧМн) | частота $\omega_0$ | Bluetooth (GFSK), модемы |
| PSK (фазовая манипуляция, ФМн) | фаза $\varphi$ | Wi-Fi, LTE, DVB-S, GPS |

QPSK - одна из самых распространенных схем модуляции: она применяется в спутниковом телевидении (DVB-S), в сотовой связи (LTE, 5G - для служебных каналов и при плохих условиях приема), в Wi-Fi (на низких скоростях), в спутниковой навигации. QPSK передает **2 бита за один символ**, имеет постоянную амплитуду символов и при этом обладает такой же помехоустойчивостью в пересчете на бит, как и простейшая двоичная BPSK.

### Структурная схема системы

Ниже приведена схема моделируемой системы. Каждый блок реализован отдельной функцией в коде.

```
ПЕРЕДАТЧИК
 биты -> [разбиение на I/Q, отображение 0->-1, 1->+1] -> [апсемплинг x N] -> [фильтр RRC] -> [квадратурный модулятор] --+
         bits_to_qpsk_symbols()                         shape_symbols()                     modulate_carrier()           |
                                                                                                                         |
КАНАЛ                                                       [шум, сдвиг фазы и частоты]  add_channel_effects()  <--------+
                                                                         |
ПРИЕМНИК                                                                 v
 биты <- [решение по знаку] <- [тактовая синхронизация] <- [согласованный фильтр RRC] <- [квадратурный демодулятор]
         symbols_to_bits()     timing_recovery()            apply_matched_filter()         demodulate_signal()
```

Обозначения, используемые далее:

| Обозначение | Смысл | Значение в коде |
| --- | --- | --- |
| $N$ | число отсчетов на символ | `samples_per_symbol = 16` |
| $T$ | длительность символа | $N$ отсчетов |
| $R_s = 1/T$ | символьная скорость | $1/16$ символа на отсчет |
| $R_b = 2R_s$ | битовая скорость | $1/8$ бита на отсчет |
| $\alpha$ | коэффициент скругления фильтра | `alpha = 0.35` |
| $f_0$ | нормированная частота несущей | `carrier_freq = 0.05` |

## 1. Квадратурное представление сигнала

Любое узкополосное колебание с меняющимися амплитудой и фазой

$$ s(t) = A(t)\cos\left(\omega_0 t + \varphi(t)\right) \tag{7.1}$$

по формуле косинуса суммы раскладывается на две составляющие:

$$ s(t) = I(t)\cos\omega_0 t - Q(t)\sin\omega_0 t, \qquad I(t) = A(t)\cos\varphi(t), \quad Q(t) = A(t)\sin\varphi(t) \tag{7.2}$$

Здесь $I(t)$ - **синфазная** составляющая (англ. *in-phase*), $Q(t)$ - **квадратурная** составляющая (англ. *quadrature*). Несущие $\cos\omega_0 t$ и $\sin\omega_0 t$ сдвинуты на $90°$ и ортогональны, поэтому по одной несущей частоте можно одновременно и независимо передавать два сигнала.

Удобно объединить $I$ и $Q$ в одно комплексное число - **комплексную огибающую** $\dot{S}(t) = I(t) + jQ(t)$:

$$ s(t) = \mathrm{Re}\left\{ \dot{S}(t)\, e^{j\omega_0 t} \right\} \tag{7.3}$$

Вся информация содержится в комплексной огибающей, а несущая лишь переносит ее на нужную частоту. Поэтому в коде символы хранятся как комплексные числа `symbols = i_symbols + 1j * q_symbols`, а на плоскости $(I, Q)$ их изображают точками - **сигнальным созвездием**.

## 2. Модуляция QPSK

В QPSK фаза несущей принимает одно из **четырех** значений: $\varphi \in \{45°, 135°, 225°, 315°\}$. Четыре состояния кодируют $\log_2 4 = 2$ бита. Поток битов $b_0, b_1, b_2, \ldots$ разбивается на пары: четные биты идут в канал $I$, нечетные - в канал $Q$:

$$ I_k = 2b_{2k} - 1, \qquad Q_k = 2b_{2k+1} - 1, \qquad \dot{S}_k = I_k + jQ_k \tag{7.4}$$

то есть бит 0 переходит в $-1$, бит 1 - в $+1$. Получается таблица отображения, реализованная в функции `bits_to_qpsk_symbols`:

| Биты $b_{2k}b_{2k+1}$ | $I_k$ | $Q_k$ | Символ $\dot{S}_k$ | Фаза |
| --- | --- | --- | --- | --- |
| 11 | +1 | +1 | $1 + j$ | $45°$ |
| 01 | -1 | +1 | $-1 + j$ | $135°$ |
| 00 | -1 | -1 | $-1 - j$ | $225°$ |
| 10 | +1 | -1 | $1 - j$ | $315°$ |

Все четыре точки лежат на окружности радиуса $\sqrt{2}$, то есть амплитуда символа постоянна, а информация передается только фазой.

**Код Грея.** Обойдите созвездие по кругу: $11 \to 01 \to 00 \to 10 \to 11$. Соседние точки отличаются ровно **одним битом**. Это важно: под действием шума символ чаще всего ошибочно принимается за *соседний*, и тогда ошибка символа дает только одну битовую ошибку, а не две. Такое отображение называется кодом Грея. Оно получается автоматически, потому что биты $I$ и $Q$ отображаются независимо.

**QPSK как две BPSK.** Из (7.4) видно, что QPSK - это два независимых двоичных сигнала BPSK ($\pm 1$), переданных на ортогональных несущих $\cos$ и $\sin$. Поэтому:

- символьная скорость вдвое меньше битовой: $R_s = R_b / 2$, и при той же полосе QPSK передает вдвое больше данных, чем BPSK;
- энергия символа делится на два бита: $E_s = 2E_b$;
- вероятность битовой ошибки QPSK и BPSK при одинаковом $E_b/N_0$ одинакова (см. раздел 6).

## 3. Формирование импульсов

### Зачем нужен формирующий фильтр

Если передавать символы прямоугольными импульсами длительностью $T$, спектр сигнала будет иметь вид $\mathrm{sinc}$ и спадать очень медленно - сигнал займет полосу соседних каналов. Чтобы ограничить спектр, каждый символ передается импульсом специальной формы $g(t)$:

$$ x(t) = \sum_k \dot{S}_k\, g(t - kT) \tag{7.5}$$

В дискретной форме это две операции (функция `shape_symbols`):

1. **апсемплинг** - между символами вставляется $N-1$ нулей (получается последовательность `upsampled`);
2. **свертка** с импульсной характеристикой формирующего КИХ-фильтра $g[n]$.

### Межсимвольная интерференция и критерий Найквиста

Импульс с ограниченным спектром неизбежно длиннее одного символа, и соседние импульсы перекрываются во времени. В момент отсчета $k$-го символа к нему добавляются «хвосты» соседних - возникает **межсимвольная интерференция (МСИ)**. Ее можно избежать, если суммарная импульсная характеристика тракта $p(t)$ удовлетворяет **критерию Найквиста**:

$$ p(nT) = \begin{cases} 1, & n = 0 \\ 0, & n \neq 0 \end{cases} \tag{7.6}$$

то есть в моменты отсчета всех соседних символов импульс проходит через ноль.

### Приподнятый косинус и корень из приподнятого косинуса

Наиболее известный импульс, удовлетворяющий (7.6), - **приподнятый косинус** (англ. *raised cosine*, RC). Его спектр плоский в центре и плавно (по косинусу) спадает к краям:

$$ P(f) = \begin{cases} T, & |f| \le \dfrac{1-\alpha}{2T} \\[2mm] \dfrac{T}{2}\left[1 + \cos\left(\dfrac{\pi T}{\alpha}\left(|f| - \dfrac{1-\alpha}{2T}\right)\right)\right], & \dfrac{1-\alpha}{2T} < |f| \le \dfrac{1+\alpha}{2T} \\[2mm] 0, & |f| > \dfrac{1+\alpha}{2T} \end{cases} \tag{7.7}$$

Параметр $\alpha \in [0, 1]$ - **коэффициент скругления** (англ. *roll-off*). В коде он задается аргументом `alpha` функций `root_raised_cosine`, `create_pulse_shape`, `shape_symbols` (передатчик) и `apply_matched_filter` (приемник), по умолчанию `alpha = 0.35`. Он задает, насколько полоса сигнала шире минимально возможной (найквистовской) $R_s/2$:

$$ B = \frac{(1+\alpha) R_s}{2} \;\;\text{(для комплексной огибающей)}, \qquad B_{\text{рад}} = (1+\alpha) R_s \;\;\text{(для сигнала на несущей)} \tag{7.8}$$

- $\alpha = 0$ - минимальная полоса, но импульс $\mathrm{sinc}$ спадает медленно, и система очень чувствительна к ошибкам синхронизации;
- $\alpha = 1$ - полоса вдвое шире, зато импульс быстро затухает, а «глаз» широко открыт.

На практике фильтрацию делят поровну между передатчиком и приемником. На каждой стороне стоит фильтр **корень из приподнятого косинуса** (англ. *root raised cosine*, RRC) с АЧХ $\sqrt{P(f)}$, и в сумме $|G(f)|^2 = P(f)$ дает приподнятый косинус, удовлетворяющий критерию Найквиста. Импульсная характеристика RRC (время $t$ измеряется в символах, как в функции `root_raised_cosine`):

$$ g(t) = \frac{\sin\left(\pi t (1-\alpha)\right) + 4\alpha t \cos\left(\pi t(1+\alpha)\right)}{\pi t \left(1 - (4\alpha t)^2\right)} \tag{7.9}$$

В точках $t = 0$ и $t = \pm\frac{1}{4\alpha}$ знаменатель обращается в ноль, и для них используются предельные значения: $g(0) = 1 - \alpha + \frac{4\alpha}{\pi}$ и $g\left(\pm\frac{1}{4\alpha}\right) = \frac{\alpha}{\sqrt 2}\left[\left(1+\frac{2}{\pi}\right)\sin\frac{\pi}{4\alpha} + \left(1-\frac{2}{\pi}\right)\cos\frac{\pi}{4\alpha}\right]$.

Фильтр RRC реализуется как КИХ-фильтр длиной $2 \cdot \text{span} \cdot N + 1$ отсчетов (`span = 6` символов в каждую сторону) и нормируется на единичную энергию $\sum g^2[n] = 1$. Обратите внимание: **сам по себе** RRC не удовлетворяет критерию Найквиста - МСИ исчезает только после второго RRC-фильтра на приемнике.

*Замечание к коду.* Свертка с `mode='same'` возвращает центральную часть результата и тем самым компенсирует задержку КИХ-фильтра в $\text{span}\cdot N$ отсчетов: пик $k$-го символа остается в отсчете $kN$. Первые и последние $\text{span}$ символов при этом искажены *краевыми эффектами* - у них обрезана часть импульса.

Ниже выполняется **шаг 1** - генерация битов, отображение в символы QPSK и формирование импульсов.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import matplotlib.gridspec as gridspec

np.random.seed(42)
plt.rcParams['figure.figsize'] = [12, 8]

def root_raised_cosine(alpha, span, sps):
    t = np.arange(-span*sps, span*sps + 1) / sps
    pulse = np.zeros_like(t)
    
    for i, t_val in enumerate(t):
        if t_val == 0:
            pulse[i] = 1.0 - alpha + (4 * alpha / np.pi)
        elif alpha != 0 and t_val == 1/(4*alpha):
            pulse[i] = (alpha/np.sqrt(2)) * ((1+2/np.pi)*np.sin(np.pi/(4*alpha)) + 
                                           (1-2/np.pi)*np.cos(np.pi/(4*alpha)))
        elif alpha != 0 and t_val == -1/(4*alpha):
            pulse[i] = (alpha/np.sqrt(2)) * ((1+2/np.pi)*np.sin(np.pi/(4*alpha)) + 
                                           (1-2/np.pi)*np.cos(np.pi/(4*alpha)))
        else:
            numerator = np.sin(np.pi*t_val*(1-alpha)) + 4*alpha*t_val*np.cos(np.pi*t_val*(1+alpha))
            denominator = np.pi*t_val*(1-(4*alpha*t_val)**2)
            pulse[i] = numerator / denominator
    
    # Нормализация
    pulse = pulse / np.sqrt(np.sum(pulse**2))
    return t, pulse

def generate_source_data(num_bits=100):
    """Генерация исходных битовых данных"""
    print("=== ШАГ 1: ГЕНЕРАЦИЯ ИСХОДНЫХ ДАННЫХ ===")
    
    # Генерация случайных битов
    source_bits = np.random.randint(0, 2, num_bits)
    print(f"Сгенерировано битов: {len(source_bits)}")
    print(f"Первые 20 битов: {source_bits[:20]}")
    
    return source_bits

def bits_to_qpsk_symbols(bits):
    """Преобразование битов в QPSK символы"""
    if len(bits) % 2 != 0:
        bits = np.append(bits, 0)  # Добавляем нуль если нечетное количество
    
    # Разделение на I и Q компоненты
    i_bits = bits[0::2]  # Четные биты -> I
    q_bits = bits[1::2]  # Нечетные биты -> Q
    
    # Преобразование в символы: 0 -> -1, 1 -> +1
    i_symbols = 2 * i_bits - 1
    q_symbols = 2 * q_bits - 1
    
    # Комплексные символы
    symbols = i_symbols + 1j * q_symbols
    
    print(f"Преобразовано в {len(symbols)} QPSK символов")
    print(f"Биты I: {i_bits[:10]} -> Символы I: {i_symbols[:10]}")
    print(f"Биты Q: {q_bits[:10]} -> Символы Q: {q_symbols[:10]}")
    print(f"Первые 5 комплексных символов: {symbols[:5]}")
    
    return symbols, i_bits, q_bits, i_symbols, q_symbols

def create_pulse_shape(samples_per_symbol=16, alpha=0.35, span=6):
    """Создание импульсной формы"""
    t, pulse = root_raised_cosine(alpha, span, samples_per_symbol)
    return t, pulse

def shape_symbols(symbols, samples_per_symbol=16, span=6, alpha=0.35):
    """Формирование импульса символов"""
    # Апсемплинг (добавление нулей между символами)
    upsampled = np.zeros(len(symbols) * samples_per_symbol, dtype=complex)
    upsampled[::samples_per_symbol] = symbols
    
    # Создание импульсной формы
    t, pulse = create_pulse_shape(samples_per_symbol, alpha=alpha, span=span)

    # Свертка с импульсной формой.
    # mode='same' компенсирует задержку фильтра (span*samples_per_symbol отсчетов),
    # поэтому пик k-го символа остается в отсчете k*samples_per_symbol
    shaped_signal = signal.convolve(upsampled, pulse, mode='same')

    print(f"Апсемплинг: {len(symbols)} символов -> {len(upsampled)} отсчетов")
    print(f"После формирования импульса: {len(shaped_signal)} отсчетов")
    
    return shaped_signal, upsampled

# Генерация и визуализация исходных данных
source_bits = generate_source_data(200)
symbols, i_bits, q_bits, i_symbols, q_symbols = bits_to_qpsk_symbols(source_bits)
shaped_signal, upsampled = shape_symbols(symbols)

# Визуализация генерации сигнала
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(3, 3)

# Исходные биты
ax1 = plt.subplot(gs[0, 0])
ax1.stem(range(20), source_bits[:20], basefmt=" ")
ax1.set_title('Исходные биты (первые 20)')
ax1.set_xlabel('Индекс бита')
ax1.set_ylabel('Значение')
ax1.grid(True)

# Созвездие QPSK
ax2 = plt.subplot(gs[0, 1])
constellation_points = [1+1j, 1-1j, -1+1j, -1-1j]
colors = ['red', 'blue', 'green', 'purple']
labels = ['11', '10', '01', '00']  # I-бит, Q-бит: 0 -> -1, 1 -> +1

for point, color, label in zip(constellation_points, colors, labels):
    ax2.scatter(np.real(point), np.imag(point), c=color, s=100, label=label)
    ax2.text(np.real(point)+0.1, np.imag(point)+0.1, label, fontsize=12)

ax2.set_title('Созвездие QPSK')
ax2.set_xlabel('I компонента')
ax2.set_ylabel('Q компонента')
ax2.legend()
ax2.grid(True)
ax2.axis('equal')

# Переданные символы
ax3 = plt.subplot(gs[0, 2])
ax3.scatter(np.real(symbols[:50]), np.imag(symbols[:50]), alpha=0.6, s=50)
ax3.set_title('Переданные символы (первые 50)')
ax3.set_xlabel('I')
ax3.set_ylabel('Q')
ax3.grid(True)

# I и Q компоненты до формирования
ax4 = plt.subplot(gs[1, 0])
time_symbols = np.arange(len(i_symbols[:20]))
ax4.stem(time_symbols, i_symbols[:20], linefmt='b-', markerfmt='bo', 
         basefmt=' ', label='I компонента')
ax4.stem(time_symbols, q_symbols[:20], linefmt='r--', markerfmt='rx', 
         basefmt=' ', label='Q компонента')
ax4.set_title('I и Q символы (первые 20)')
ax4.set_xlabel('Индекс символа')
ax4.set_ylabel('Амплитуда')
ax4.legend()
ax4.grid(True)

# Импульсная форма
ax5 = plt.subplot(gs[1, 1])
t, pulse = create_pulse_shape()
ax5.plot(t, pulse, 'g-', linewidth=2)
ax5.set_title('Импульсная форма\n(корень из приподнятого косинуса)')
ax5.set_xlabel('Отсчеты')
ax5.set_ylabel('Амплитуда')
ax5.grid(True)

# Сформированный сигнал
ax6 = plt.subplot(gs[1, 2])
time_shaped = np.arange(200)
ax6.plot(time_shaped, np.real(shaped_signal)[:200], 'b-', label='I компонента', alpha=0.8)
ax6.plot(time_shaped, np.imag(shaped_signal)[:200], 'r-', label='Q компонента', alpha=0.8)
ax6.set_title('Сформированный сигнал (первые 200 отсчетов)')
ax6.set_xlabel('Отсчеты')
ax6.set_ylabel('Амплитуда')
ax6.legend()
ax6.grid(True)

plt.tight_layout()
plt.show()

## 4. Квадратурный модулятор и канал связи

### Перенос на несущую

Функция `modulate_carrier` реализует формулу (7.2) в дискретном времени:

$$ s[n] = I[n]\cos(2\pi f_0 n) - Q[n]\sin(2\pi f_0 n) \tag{7.10}$$

где $f_0$ - частота несущей, нормированная к частоте дискретизации (`carrier_freq = 0.05`, то есть 20 отсчетов на период несущей). Спектр комплексной огибающей переносится на частоты $\pm f_0$. Чтобы положительная и отрицательная части спектра не перекрывались, несущая должна быть больше полосы огибающей (7.8): $f_0 > \frac{1+\alpha}{2N}$. Для $\alpha = 0.35$, $N = 16$ это $0.042$ - условие выполнено.

### Канал с аддитивным белым гауссовым шумом (АБГШ)

Простейшая модель канала - прибавление белого гауссова шума с дисперсией $\sigma^2$:

$$ r[n] = s[n] + w[n], \qquad w[n] \sim \mathcal{N}(0, \sigma^2) \tag{7.11}$$

В функции `add_channel_effects` шум задается через **отношение сигнал/шум** на отсчет:

$$ \mathrm{SNR} = \frac{P_s}{\sigma^2}, \qquad P_s = \overline{s^2[n]} \tag{7.12}$$

Однако системы связи принято сравнивать по отношению **энергии бита к спектральной плотности шума** $E_b/N_0$ - эта величина не зависит от числа отсчетов на символ и частоты дискретизации. Для нашей модели (вещественный сигнал на несущей, $N$ отсчетов на символ, 2 бита на символ) энергия символа $E_s = P_s N$, энергия бита $E_b = E_s/2$, а двусторонняя спектральная плотность белого шума $N_0/2 = \sigma^2$. Отсюда

$$ \frac{E_b}{N_0} = \mathrm{SNR}\cdot\frac{N}{4}, \qquad \mathrm{SNR}_{\text{дБ}} = \left(\frac{E_b}{N_0}\right)_{\text{дБ}} - 10\lg\frac{N}{4} \tag{7.13}$$

При $N = 16$: $\mathrm{SNR}_{\text{дБ}} = (E_b/N_0)_{\text{дБ}} - 6.02$ дБ. Например, `snr_db = 15` соответствует $E_b/N_0 \approx 21$ дБ.

### Фазовая и частотная расстройка

Генераторы несущей передатчика и приемника никогда не совпадают точно. Если несущая приемника отличается по фазе на $\varphi$ и по частоте на $\Delta f$, то относительно приемника принятый сигнал выглядит так:

$$ r(t) = I(t)\cos\left(\omega_0 t + \theta(t)\right) - Q(t)\sin\left(\omega_0 t + \theta(t)\right), \qquad \theta(t) = 2\pi\Delta f\, t + \varphi \tag{7.14}$$

В терминах комплексной огибающей это просто поворот:

$$ \dot{R}(t) = \dot{S}(t)\, e^{j\theta(t)} \tag{7.15}$$

- постоянный сдвиг фазы $\varphi$ **поворачивает созвездие** на угол $\varphi$;
- сдвиг частоты $\Delta f$ заставляет созвездие **непрерывно вращаться**: за один символ оно поворачивается на $360° \cdot \Delta f \cdot N$.

В реальных приемниках эти сдвиги устраняет схема **восстановления несущей** (петля Костаса, ФАПЧ). В этой работе она не реализована, поэтому расстройка напрямую влияет на результат. В коде поворот (7.15) реализован через аналитический сигнал (`signal.hilbert`).

Ниже выполняется **шаг 2** - перенос на несущую и прохождение через канал.

In [ ]:
def modulate_carrier(shaped_signal, carrier_freq=0.05, sample_rate=1.0):
    """Модуляция на несущую частоту"""
    print("\n=== ШАГ 2: МОДУЛЯЦИЯ НА НЕСУЩУЮ ===")
    
    t = np.arange(len(shaped_signal)) / sample_rate
    
    # Разделение на I и Q компоненты
    i_signal = np.real(shaped_signal)
    q_signal = np.imag(shaped_signal)
    
    # Квадратурная модуляция
    carrier_i = i_signal * np.cos(2 * np.pi * carrier_freq * t)
    carrier_q = q_signal * np.sin(2 * np.pi * carrier_freq * t)
    
    # Суммирование для получения вещественного сигнала
    modulated_signal = carrier_i - carrier_q  # Минус для правильной фазы
    
    print(f"Несущая частота: {carrier_freq:.3f} (относительная)")
    print(f"Длина модулированного сигнала: {len(modulated_signal)} отсчетов")
    
    return modulated_signal, t, carrier_i, carrier_q

def add_channel_effects(tx_signal, snr_db=20, frequency_offset=0.0, phase_offset=0.0):
    """Добавление эффектов канала связи"""
    # Добавление шума
    signal_power = np.mean(tx_signal**2)
    noise_power = signal_power / (10**(snr_db/10))
    noise = np.sqrt(noise_power) * np.random.randn(len(tx_signal))

    # Частотный и фазовый сдвиг несущей: cos(w*t) -> cos((w + 2*pi*df)*t + phi).
    # Через аналитический сигнал это поворот на угол 2*pi*df*t + phi
    t = np.arange(len(tx_signal))
    rotation = np.exp(1j * (2 * np.pi * frequency_offset * t + phase_offset))
    shifted = np.real(signal.hilbert(tx_signal) * rotation)

    impaired_signal = shifted + noise

    print(f"Добавлен шум: SNR = {snr_db} dB")
    print(f"Частотный сдвиг: {frequency_offset:.4f}")
    print(f"Фазовый сдвиг: {phase_offset:.3f} рад")
    
    return impaired_signal, noise

# Модуляция и добавление эффектов канала
modulated, time_vec, carrier_i, carrier_q = modulate_carrier(shaped_signal)
received_signal, channel_noise = add_channel_effects(modulated, snr_db=15, 
                                                   frequency_offset=0.0, 
                                                   phase_offset=0.02)

# Визуализация модуляции
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(3, 2)

# I и Q компоненты перед модуляцией
ax1 = plt.subplot(gs[0, 0])
ax1.plot(time_vec[:200], np.real(shaped_signal)[:200], 'b-', label='I компонента', linewidth=2)
ax1.plot(time_vec[:200], np.imag(shaped_signal)[:200], 'r-', label='Q компонента', linewidth=2)
ax1.set_title('I и Q компоненты до модуляции')
ax1.set_xlabel('Время')
ax1.set_ylabel('Амплитуда')
ax1.legend()
ax1.grid(True)

# Несущие колебания
ax2 = plt.subplot(gs[0, 1])
t_short = time_vec[:100]
ax2.plot(t_short, np.cos(2*np.pi*0.05*t_short), 'b-', label='cos(ωt) - для I', alpha=0.7)
ax2.plot(t_short, np.sin(2*np.pi*0.05*t_short), 'r-', label='sin(ωt) - для Q', alpha=0.7)
ax2.set_title('Несущие колебания')
ax2.set_xlabel('Время')
ax2.set_ylabel('Амплитуда')
ax2.legend()
ax2.grid(True)

# Модулированные компоненты
ax3 = plt.subplot(gs[1, 0])
ax3.plot(time_vec[:200], carrier_i[:200], 'b-', label='I × cos(ωt)', alpha=0.8)
ax3.plot(time_vec[:200], carrier_q[:200], 'r-', label='Q × sin(ωt)', alpha=0.8)
ax3.set_title('Модулированные I и Q компоненты')
ax3.set_xlabel('Время')
ax3.set_ylabel('Амплитуда')
ax3.legend()
ax3.grid(True)

# Полный модулированный сигнал
ax4 = plt.subplot(gs[1, 1])
ax4.plot(time_vec[:200], modulated[:200], 'g-', linewidth=2)
ax4.set_title('Полный модулированный сигнал\n(I×cos(ωt) - Q×sin(ωt))')
ax4.set_xlabel('Время')
ax4.set_ylabel('Амплитуда')
ax4.grid(True)

# Сигнал после канала
ax5 = plt.subplot(gs[2, 0])
ax5.plot(time_vec[:200], received_signal[:200], 'g-', label='Принятый сигнал', alpha=0.8)
ax5.plot(time_vec[:200], modulated[:200], 'k--', label='Идеальный сигнал', alpha=0.6)
ax5.set_title('Сигнал после канала связи')
ax5.set_xlabel('Время')
ax5.set_ylabel('Амплитуда')
ax5.legend()
ax5.grid(True)

# Шум канала
ax6 = plt.subplot(gs[2, 1])
ax6.plot(time_vec[:200], channel_noise[:200], 'm-', alpha=0.7)
ax6.set_title('Шум канала связи')
ax6.set_xlabel('Время')
ax6.set_ylabel('Амплитуда')
ax6.grid(True)

plt.tight_layout()
plt.show()

## 5. Приемник

### Квадратурный демодулятор

Чтобы выделить $I$ и $Q$, принятый сигнал умножается на копии несущих (функция `demodulate_signal`). Используя формулы произведения тригонометрических функций, получаем:

$$ 2s(t)\cos\omega_0 t = I(t) + \underbrace{I(t)\cos 2\omega_0 t - Q(t)\sin 2\omega_0 t}_{\text{на частоте } 2f_0} $$

$$ -2s(t)\sin\omega_0 t = Q(t) \underbrace{- Q(t)\cos 2\omega_0 t - I(t)\sin 2\omega_0 t}_{\text{на частоте } 2f_0} \tag{7.16}$$

Множитель 2 компенсирует коэффициент $1/2$, который возникает при перемножении. Составляющие на удвоенной частоте $2f_0 = 0.1$ нужно подавить фильтром нижних частот. Отдельный ФНЧ здесь не нужен: его роль выполняет согласованный фильтр, полоса которого $\frac{1+\alpha}{2N}$ много меньше $2f_0$.

### Согласованный фильтр

**Согласованным** называется фильтр, импульсная характеристика которого - зеркальная копия ожидаемого импульса:

$$ h_{\text{СФ}}(t) = g(T_0 - t) \tag{7.17}$$

Среди всех линейных фильтров согласованный дает **максимальное отношение сигнал/шум** в момент отсчета: оно равно $2E/N_0$, где $E$ - энергия импульса. Импульс RRC симметричен, поэтому согласованный фильтр совпадает с формирующим (функция `apply_matched_filter` использует тот же `create_pulse_shape`). Последовательное соединение двух RRC-фильтров дает приподнятый косинус, и в моменты отсчета МСИ отсутствует.

### Тактовая синхронизация

После согласованного фильтра сигнал нужно взять в один отсчет на символ - в момент максимального раскрыва «глаза». Приемник заранее не знает, какой из $N$ отсчетов внутри символа оптимален. В функции `timing_recovery` реализован простой алгоритм: перебираются все сдвиги $d = 0, \ldots, N-1$, и выбирается тот, при котором средняя мощность в точках отсчета максимальна:

$$ \hat{d} = \arg\max_{d} \sum_k \left|y[d + kN]\right|^2 \tag{7.18}$$

Идея: в правильные моменты отсчета сигнал принимает значения $\pm 1$, а между ними из-за перехода от символа к символу его модуль в среднем меньше. В реальных системах используют адаптивные алгоритмы, которые отслеживают дрейф тактовой частоты, например алгоритмы Гарднера или Мюллера-Мюллера.

### Глазковая диаграмма

**Глазковая диаграмма** - это наложенные друг на друга отрезки сигнала длиной в несколько символов, синхронизированные с тактовой частотой. Ее вид позволяет сразу оценить качество тракта:

- **вертикальный раскрыв** глаза - запас помехоустойчивости по амплитуде (шум и МСИ «закрывают» глаз);
- **горизонтальный раскрыв** - допустимая ошибка момента отсчета;
- оптимальный момент отсчета - точка максимального раскрыва.

Ниже выполняется **шаг 3** - демодуляция, согласованная фильтрация и тактовая синхронизация.

In [ ]:
def demodulate_signal(received_signal, carrier_freq=0.05, sample_rate=1.0):
    """Квадратурная демодуляция сигнала"""
    print("\n=== ШАГ 3: ДЕМОДУЛЯЦИЯ СИГНАЛА ===")
    
    t = np.arange(len(received_signal)) / sample_rate
    
    # Квадратурное детектирование
    i_demod = 2 * received_signal * np.cos(2 * np.pi * carrier_freq * t)
    q_demod = -2 * received_signal * np.sin(2 * np.pi * carrier_freq * t)  # Минус для компенсации
    
    print(f"Демодулированные I компонента: {len(i_demod)} отсчетов")
    print(f"Демодулированные Q компонента: {len(q_demod)} отсчетов")
    
    return i_demod, q_demod, t

def apply_matched_filter(i_signal, q_signal, samples_per_symbol=16, span=6, alpha=0.35):
    """Применение согласованного фильтра"""
    # Создание импульсной формы (такой же как на передаче)
    _, pulse = create_pulse_shape(samples_per_symbol, alpha=alpha, span=span)

    # Фильтрация I и Q компонент (mode='same' компенсирует задержку фильтра)
    i_filtered = signal.convolve(i_signal, pulse, mode='same')
    q_filtered = signal.convolve(q_signal, pulse, mode='same')

    print(f"После согласованной фильтрации: {len(i_filtered)} отсчетов")
    
    return i_filtered, q_filtered

def timing_recovery(i_filtered, q_filtered, samples_per_symbol=16):
    """Тактовая синхронизация"""
    # Объединение I и Q для определения времени
    power_signal = i_filtered**2 + q_filtered**2
    
    # Поиск оптимальной фазы отсчета: для каждого сдвига d = 0..sps-1
    # считаем среднюю мощность в точках d, d+sps, d+2*sps, ...
    # и выбираем сдвиг с максимальной средней мощностью (центр "глаза")
    mean_power = [np.mean(power_signal[d::samples_per_symbol])
                  for d in range(samples_per_symbol)]
    best_offset = int(np.argmax(mean_power))
    symbol_indices = list(range(best_offset, len(power_signal), samples_per_symbol))
    print(f"Оптимальная фаза отсчета: {best_offset} (из {samples_per_symbol})")

    # Извлечение символов в оптимальные моменты времени
    i_symbols = i_filtered[symbol_indices]
    q_symbols = q_filtered[symbol_indices]
    recovered_symbols = i_symbols + 1j * q_symbols
    
    print(f"Восстановлено символов: {len(recovered_symbols)}")
    
    return recovered_symbols, symbol_indices, power_signal

# Демодуляция
i_demod, q_demod, t_demod = demodulate_signal(received_signal)
i_filtered, q_filtered = apply_matched_filter(i_demod, q_demod)
recovered_symbols, symbol_times, power_signal = timing_recovery(i_filtered, q_filtered)

# Визуализация демодуляции
fig = plt.figure(figsize=(16, 14))
gs = gridspec.GridSpec(4, 2)

# Демодулированные I и Q
ax1 = plt.subplot(gs[0, 0])
ax1.plot(t_demod[:300], i_demod[:300], 'b-', alpha=0.7, label='I демодулированная')
ax1.plot(t_demod[:300], q_demod[:300], 'r-', alpha=0.7, label='Q демодулированная')
ax1.set_title('Демодулированные I и Q компоненты')
ax1.set_xlabel('Время')
ax1.set_ylabel('Амплитуда')
ax1.legend()
ax1.grid(True)

# После согласованного фильтра
ax2 = plt.subplot(gs[0, 1])
ax2.plot(t_demod[:300], i_filtered[:300], 'b-', linewidth=2, label='I после фильтра')
ax2.plot(t_demod[:300], q_filtered[:300], 'r-', linewidth=2, label='Q после фильтра')
ax2.set_title('После согласованной фильтрации')
ax2.set_xlabel('Время')
ax2.set_ylabel('Амплитуда')
ax2.legend()
ax2.grid(True)

# Глазковая диаграмма для I компоненты
ax3 = plt.subplot(gs[1, 0])
samples_per_symbol = 16
for i in range(min(10, len(i_filtered)//samples_per_symbol)):
    start_idx = i * samples_per_symbol
    end_idx = start_idx + 2 * samples_per_symbol
    if end_idx < len(i_filtered):
        eye_segment = i_filtered[start_idx:end_idx]
        ax3.plot(np.arange(len(eye_segment)), eye_segment, 'b-', alpha=0.5)
ax3.set_title('Глазковая диаграмма (I компонента)')
ax3.set_xlabel('Отсчеты')
ax3.set_ylabel('Амплитуда')
ax3.grid(True)

# Восстановленное созвездие
ax5 = plt.subplot(gs[1, 1])
ax5.scatter(np.real(recovered_symbols), np.imag(recovered_symbols), 
           c='red', alpha=0.6, s=50, label='Восстановленные')
ax5.scatter(np.real(symbols[:len(recovered_symbols)]), 
           np.imag(symbols[:len(recovered_symbols)]), 
           c='blue', alpha=0.3, s=30, label='Исходные')
ax5.set_title('Сравнение созвездий')
ax5.set_xlabel('I компонента')
ax5.set_ylabel('Q компонента')
ax5.legend()
ax5.grid(True)

# Сравнение I компонент
ax6 = plt.subplot(gs[2, 0])
symbol_indices = range(min(20, len(recovered_symbols)))
ax6.stem(symbol_indices, np.real(symbols[:len(recovered_symbols)][:20]), 
         linefmt='b-', markerfmt='bo', basefmt=' ', label='Исходные I')
ax6.stem(symbol_indices, np.real(recovered_symbols[:20]), 
         linefmt='r--', markerfmt='rx', basefmt=' ', label='Восстановленные I')
ax6.set_title('Сравнение I компонент символов')
ax6.set_xlabel('Индекс символа')
ax6.set_ylabel('Амплитуда')
ax6.legend()
ax6.grid(True)

# Сравнение Q компонент
ax7 = plt.subplot(gs[2, 1])
ax7.stem(symbol_indices, np.imag(symbols[:len(recovered_symbols)][:20]), 
         linefmt='b-', markerfmt='bo', basefmt=' ', label='Исходные Q')
ax7.stem(symbol_indices, np.imag(recovered_symbols[:20]), 
         linefmt='r--', markerfmt='rx', basefmt=' ', label='Восстановленные Q')
ax7.set_title('Сравнение Q компонент символов')
ax7.set_xlabel('Индекс символа')
ax7.set_ylabel('Амплитуда')
ax7.legend()
ax7.grid(True)

plt.tight_layout()
plt.show()

## 6. Принятие решения и оценка качества

### Решающее устройство

Благодаря коду Грея и независимости $I$ и $Q$ оптимальное решение принимается отдельно по знаку каждой компоненты (функция `symbols_to_bits`). Границы решений - оси координат, которые делят плоскость на четыре квадранта:

$$ \hat{b}_{2k} = \begin{cases} 1, & \mathrm{Re}\,y_k > 0 \\ 0, & \mathrm{Re}\,y_k \le 0 \end{cases}, \qquad \hat{b}_{2k+1} = \begin{cases} 1, & \mathrm{Im}\,y_k > 0 \\ 0, & \mathrm{Im}\,y_k \le 0 \end{cases} \tag{7.19}$$

### Показатели качества

- **BER** (*Bit Error Rate*) - доля ошибочно принятых битов.
- **SER** (*Symbol Error Rate*) - доля ошибочно принятых символов (хотя бы один бит из двух неверен).
- **EVM** (*Error Vector Magnitude*) - среднеквадратичная величина вектора ошибки между принятым и переданным символом:

$$ \mathrm{EVM} = \sqrt{\frac{1}{K}\sum_{k=1}^{K}\left|y_k - \dot{S}_k\right|^2} \tag{7.20}$$

EVM полезна тем, что показывает искажения (шум, МСИ, поворот фазы) даже тогда, когда ошибок еще нет. В стандартах EVM обычно выражают в процентах от среднеквадратичной амплитуды созвездия; для нашего созвездия она равна $\sqrt 2$, поэтому $\mathrm{EVM}_{\%} = \mathrm{EVM}/\sqrt{2} \cdot 100\%$.

### Теоретическая помехоустойчивость

Для канала с АБГШ, идеальной синхронизацией и кодом Грея вероятность битовой ошибки QPSK равна

$$ P_b = Q\left(\sqrt{\frac{2E_b}{N_0}}\right) = \frac{1}{2}\,\mathrm{erfc}\left(\sqrt{\frac{E_b}{N_0}}\right), \qquad P_s = 1 - (1 - P_b)^2 \approx 2P_b \tag{7.21}$$

где $Q(x) = \frac{1}{\sqrt{2\pi}}\int_x^\infty e^{-u^2/2}du$ - хвост нормального распределения, $\mathrm{erfc}$ - дополнительная функция ошибок (`scipy.special.erfc`). Формула совпадает с формулой для BPSK, потому что каналы $I$ и $Q$ - это две независимые BPSK.

Для справки - несколько значений (7.21):

| $E_b/N_0$, дБ | 0 | 2 | 4 | 6 | 8 | 10 |
| --- | --- | --- | --- | --- | --- | --- |
| $P_b$ | $7.9\cdot10^{-2}$ | $3.8\cdot10^{-2}$ | $1.3\cdot10^{-2}$ | $2.4\cdot10^{-3}$ | $1.9\cdot10^{-4}$ | $3.9\cdot10^{-6}$ |

Чтобы надежно оценить BER порядка $10^{-p}$, нужно передать не меньше $10^{p+1}-10^{p+2}$ битов (примерно 10-100 ошибок).

Ниже выполняется **шаг 4** - восстановление битов и расчет BER, SER, EVM.

In [ ]:
def symbols_to_bits(symbols, decision_threshold=0):
    """Преобразование символов обратно в биты"""
    # Принятие решения по символам
    i_decisions = (np.real(symbols) > decision_threshold).astype(int)
    q_decisions = (np.imag(symbols) > decision_threshold).astype(int)
    
    # Объединение битов
    recovered_bits = np.zeros(2 * len(i_decisions))
    recovered_bits[0::2] = i_decisions
    recovered_bits[1::2] = q_decisions
    
    print(f"Восстановлено битов: {len(recovered_bits)}")
    print(f"Первые 20 восстановленных битов: {recovered_bits[:20].astype(int)}")
    
    return recovered_bits, i_decisions, q_decisions

def calculate_performance(original_bits, recovered_bits, original_symbols, recovered_symbols):
    """Расчет характеристик системы"""
    # Bit Error Rate (BER)
    min_len = min(len(original_bits), len(recovered_bits))
    bit_errors = np.sum(original_bits[:min_len] != recovered_bits[:min_len])
    ber = bit_errors / min_len
    
    # Symbol Error Rate (SER)
    min_symbols = min(len(original_symbols), len(recovered_symbols))
    symbol_errors = 0
    for i in range(min_symbols):
        original_i = 1 if np.real(original_symbols[i]) > 0 else 0
        original_q = 1 if np.imag(original_symbols[i]) > 0 else 0
        recovered_i = 1 if np.real(recovered_symbols[i]) > 0 else 0
        recovered_q = 1 if np.imag(recovered_symbols[i]) > 0 else 0
        
        if original_i != recovered_i or original_q != recovered_q:
            symbol_errors += 1
    
    ser = symbol_errors / min_symbols
    
    # Error Vector Magnitude (EVM)
    evm = np.sqrt(np.mean(np.abs(original_symbols[:min_symbols] - recovered_symbols[:min_symbols])**2))
    
    print("\n=== ПРОИЗВОДИТЕЛЬНОСТЬ СИСТЕМЫ ===")
    print(f"Bit Error Rate (BER): {ber:.6f} ({bit_errors} ошибок из {min_len} битов)")
    print(f"Symbol Error Rate (SER): {ser:.6f} ({symbol_errors} ошибок из {min_symbols} символов)")
    print(f"Error Vector Magnitude (EVM): {evm:.6f}")
    
    return ber, ser, evm, bit_errors, symbol_errors

# Восстановление битов
recovered_bits, i_decisions, q_decisions = symbols_to_bits(recovered_symbols)

# Расчет характеристик
ber, ser, evm, bit_errors, symbol_errors = calculate_performance(
    source_bits, recovered_bits, symbols, recovered_symbols)

# Детальное сравнение данных
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(3, 2)

# Сравнение всех битов
ax1 = plt.subplot(gs[0, :])
bit_indices = np.arange(min(100, len(source_bits), len(recovered_bits)))
ax1.stem(bit_indices, source_bits[:len(bit_indices)], 
         linefmt='b-', markerfmt='bo', basefmt=' ', label='Переданные биты')
ax1.stem(bit_indices, recovered_bits[:len(bit_indices)], 
         linefmt='r--', markerfmt='rx', basefmt=' ', label='Принятые биты')

# Подсветка ошибок
error_indices = bit_indices[source_bits[:len(bit_indices)] != recovered_bits[:len(bit_indices)]]
ax1.plot(error_indices, np.ones_like(error_indices)*1.1, 'rv', 
         markersize=10, label='Ошибки', alpha=0.7)

ax1.set_title('ДЕТАЛЬНОЕ СРАВНЕНИЕ ПЕРЕДАННЫХ И ПРИНЯТЫХ БИТОВ', fontsize=14, fontweight='bold')
ax1.set_xlabel('Индекс бита')
ax1.set_ylabel('Значение бита')
ax1.legend()
ax1.grid(True)
ax1.set_ylim(-0.1, 1.4)

# Таблица ошибок
ax2 = plt.subplot(gs[1, 0])
ax2.axis('off')
error_table_data = [
    ["Параметр", "Значение"],
    ["Всего битов", f"{min(len(source_bits), len(recovered_bits))}"],
    ["Ошибок битов", f"{bit_errors}"],
    ["BER", f"{ber:.6f}"],
    ["Ошибок символов", f"{symbol_errors}"],
    ["SER", f"{ser:.6f}"],
    ["EVM", f"{evm:.6f}"]
]

table = ax2.table(cellText=error_table_data, 
                 cellLoc='center', 
                 loc='center',
                 bbox=[0.1, 0.1, 0.8, 0.8])
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2)
ax2.set_title('СТАТИСТИКА ОШИБОК', fontweight='bold')

# Сравнение созвездий с ошибками
ax3 = plt.subplot(gs[1, 1])
# Создаем маски для правильных и ошибочных символов
correct_mask = np.zeros(len(recovered_symbols), dtype=bool)
for i in range(min(len(recovered_symbols), len(source_bits)//2)):
    original_i = 1 if np.real(symbols[i]) > 0 else 0
    original_q = 1 if np.imag(symbols[i]) > 0 else 0
    recovered_i = 1 if np.real(recovered_symbols[i]) > 0 else 0
    recovered_q = 1 if np.imag(recovered_symbols[i]) > 0 else 0
    
    correct_mask[i] = (original_i == recovered_i) and (original_q == recovered_q)

correct_symbols = recovered_symbols[correct_mask]
error_symbols = recovered_symbols[~correct_mask]

ax3.scatter(np.real(correct_symbols), np.imag(correct_symbols), 
           c='green', alpha=0.6, s=50, label='Правильные символы')
ax3.scatter(np.real(error_symbols), np.imag(error_symbols), 
           c='red', alpha=0.8, s=80, label='Ошибочные символы', marker='x')

# Исходное созвездие
for point, color, label in zip(constellation_points, colors, labels):
    ax3.scatter(np.real(point), np.imag(point), c=color, s=200, alpha=0.3)
    ax3.text(np.real(point)+0.1, np.imag(point)+0.1, label, fontsize=12)

ax3.set_title('Созвездие с выделением ошибок')
ax3.set_xlabel('I компонента')
ax3.set_ylabel('Q компонента')
ax3.legend()
ax3.grid(True)
ax3.axis('equal')

# Распределение ошибок по времени
ax4 = plt.subplot(gs[2, :])
symbol_time_indices = np.arange(len(recovered_symbols))
error_mask = ~correct_mask

ax4.plot(symbol_time_indices, np.abs(recovered_symbols), 'b-', 
         alpha=0.5, label='Амплитуда символов')
ax4.plot(symbol_time_indices[error_mask], np.abs(recovered_symbols[error_mask]), 'ro',
         markersize=8, label='Ошибочные символы')

ax4.set_title('РАСПРЕДЕЛЕНИЕ ОШИБОК ВО ВРЕМЕНИ', fontweight='bold')
ax4.set_xlabel('Индекс символа')
ax4.set_ylabel('Амплитуда символа')
ax4.legend()
ax4.grid(True)

plt.tight_layout()
plt.show()

# Итоговый вывод
print("\n" + "="*60)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("="*60)
print(f"✓ Успешно передано и восстановлено {min(len(source_bits), len(recovered_bits))} битов")
print(f"✓ Количество битовых ошибок: {bit_errors}")
print(f"✓ Bit Error Rate: {ber:.2%}")
print(f"✓ Качество связи: {'ОТЛИЧНО' if ber < 0.01 else 'ХОРОШО' if ber < 0.05 else 'УДОВЛЕТВОРИТЕЛЬНО'}")
print("="*60)

# Задания

**Общие указания.**

- Перед выполнением заданий запустите все ячейки сверху вниз и разберитесь, какая функция за что отвечает (см. структурную схему во введении).
- Задания выполняются **небольшими правками** готового кода: изменением параметров в вызовах функций, добавлением цикла или нескольких строк. Переписывать функции не требуется.
- Для каждого задания добавьте под ним новые ячейки с кодом и ячейку с выводами. Исходные ячейки можно менять, но после выполнения задания верните параметры по умолчанию.
- Функции печатают промежуточную информацию. Если в цикле это мешает, оберните вызов в `with contextlib.redirect_stdout(io.StringIO()):` (модули `io` и `contextlib`).
- Все графики подписывайте: заголовок, оси, легенда.

### Задание 1

**Проверка системы в идеальном канале. Краевые эффекты.**

Требуется:

1. В вызове `add_channel_effects` (шаг 2) задать `snr_db=100`, `phase_offset=0` и перезапустить шаги 2-4. Убедиться, что BER = SER = 0 и созвездие на приемнике сжалось в четыре точки. Записать значение EVM.
2. Увеличить число битов в `generate_source_data` с 200 до 2000 и повторить расчет. Как изменилось EVM?
3. Посчитать EVM без первых и последних `span = 6` символов (одна строка кода по формуле (7.20)).
4. Свести результаты в таблицу:

| Число битов | EVM по всем символам | EVM без краевых символов |
| --- | --- | --- |
| 200 | | |
| 2000 | | |

5. Объяснить, почему EVM в идеальном канале не равна нулю и почему она уменьшается с ростом числа битов.

*Подсказки:*

- переданные символы - `symbols`, принятые - `recovered_symbols`; срез без краев: `x[6:-6]`;
- вспомните замечание о `mode='same'` в разделе 3.

### Задание 2

**Влияние шума.**

Число битов - 2000, `phase_offset=0`, `frequency_offset=0`.

Требуется:

1. Выполнить шаги 2-4 для `snr_db` = 20, 10, 5 и 0 дБ.
2. Для каждого значения пересчитать SNR в $E_b/N_0$ по формуле (7.13) и сохранить график созвездия (шаг 3, «Сравнение созвездий») и глазковую диаграмму.
3. Свести в таблицу: SNR, дБ / $E_b/N_0$, дБ / BER / SER / EVM / теоретическая BER по формуле (7.21).
4. Ответить на вопросы:
   - при каком SNR появляются первые ошибки?
   - почему облако точек вокруг каждой точки созвездия имеет форму круга?
   - как шум меняет глазковую диаграмму?
   - выполняется ли приближенное соотношение $\mathrm{SER} \approx 2\,\mathrm{BER}$?

### Задание 3

**Кривая помехоустойчивости BER($E_b/N_0$).**

Требуется:

1. Для $E_b/N_0$ = 0, 1, ..., 8 дБ промоделировать передачу $2\cdot10^5$ битов и посчитать BER. Фазовая и частотная расстройка - нулевые.
2. Построить на одном графике в логарифмическом масштабе по оси BER (`plt.semilogy`) экспериментальную кривую (маркеры) и теоретическую по формуле (7.21) (сплошная линия).
3. Оценить, насколько (в дБ по оси $E_b/N_0$) экспериментальная кривая отличается от теоретической, и объяснить причины расхождений, если они есть.
4. Объяснить, почему при $E_b/N_0 = 10$ дБ надежно измерить BER таким числом битов не получится.

*Подсказка* - заготовка цикла, в которой нужно дописать две строки:

```python
from scipy.special import erfc

sps = 16
ebn0_db = np.arange(0, 9)
ber_sim = []

for eb in ebn0_db:
    snr_db = ...                                   # пересчет по формуле (7.13)
    bits = generate_source_data(200000)
    sym, *_ = bits_to_qpsk_symbols(bits)
    shaped, _ = shape_symbols(sym)
    tx, *_ = modulate_carrier(shaped)
    rx, _ = add_channel_effects(tx, snr_db=snr_db)
    i_d, q_d, _ = demodulate_signal(rx)
    i_f, q_f = apply_matched_filter(i_d, q_d)
    rec, _, _ = timing_recovery(i_f, q_f)
    rec_bits, *_ = symbols_to_bits(rec)
    ber, *_ = calculate_performance(bits, rec_bits, sym, rec)
    ber_sim.append(ber)

ber_theory = ...                                   # формула (7.21)
```

### Задание 4

**Влияние коэффициента скругления $\alpha$.**

Функции `shape_symbols` и `apply_matched_filter` принимают параметр `alpha` - коэффициент скругления $\alpha$. На передатчике и приемнике он должен быть одинаковым. Число битов - 2000, `snr_db = 100`.

Требуется для $\alpha$ = 0.1, 0.35 и 0.9:

1. Построить на одном графике импульсные характеристики RRC (`create_pulse_shape(alpha=...)`).
2. Построить на одном графике спектральную плотность мощности сигнала на несущей `modulated` в дБ:
   `f, Pxx = signal.welch(modulated, fs=1.0, nperseg=256)`, затем `10*np.log10(Pxx / Pxx.max())`.
3. По графику измерить ширину спектра по уровню $-20$ дБ и сравнить с теоретической $(1+\alpha)R_s$ из (7.8), где $R_s = 1/16$.
4. Построить глазковые диаграммы (в шаге 3 увеличьте число наложенных отрезков с 10 до 100) и сравнить горизонтальный раскрыв глаза.
5. Проверить для каждого $\alpha$ условие $f_0 > \frac{1+\alpha}{2N}$ из раздела 4. Для какого $\alpha$ оно нарушено и как это отражается на EVM?
6. Свести в таблицу: $\alpha$ / теоретическая полоса / измеренная полоса / EVM. Почему при $\alpha = 0.1$ EVM тоже выше, чем при $\alpha = 0.35$? (Подсказка: вспомните, что фильтр обрезан до `span = 6` символов, и посмотрите на импульсную характеристику из п. 1.)
7. Сделать вывод о компромиссе между полосой и устойчивостью к ошибкам синхронизации.

### Задание 5

**Фазовая и частотная расстройка несущей.**

Число битов - 2000, `snr_db = 20`.

Требуется:

1. При `frequency_offset=0` задать фазовую ошибку `phase_offset = np.deg2rad(phi)` для $\varphi$ = 0°, 15°, 30°, 45°, 60°. Для каждого значения построить созвездие и записать BER и EVM в таблицу.
2. Объяснить, используя (7.15) и (7.19):
   - почему при $\varphi < 45°$ ошибок почти нет, хотя EVM растет;
   - почему при $\varphi = 45°$ BER $\approx 0.25$;
   - что происходит при $\varphi > 45°$.
3. **Компенсация фазы.** При $\varphi = 60°$ перед вызовом `symbols_to_bits` повернуть принятые символы обратно, добавив одну строку:
   `recovered_symbols = recovered_symbols * np.exp(-1j * np.deg2rad(60))`.
   Убедиться, что BER вернулась к нулю. Почему в реальном приемнике так сделать нельзя и что для этого используют?
4. При `phase_offset=0` задать `frequency_offset=0.0005`. Построить созвездие и график набега фазы от номера символа. Чтобы убрать из фазы информационную составляющую (кратную $90°$), возведите символы в четвертую степень: `np.unwrap(np.angle(recovered_symbols**4)) / 4`. Рассчитать теоретический поворот за символ ($360° \cdot \Delta f \cdot N$) и сравнить с наклоном графика.

### Задание 6 (дополнительное)

**Ошибка тактовой синхронизации.**

Требуется:

1. Добавить функции `timing_recovery` параметр `delta=0`, а между строками `best_offset = ...` и `symbol_indices = ...` - искусственный сдвиг момента отсчета: `best_offset = best_offset + delta`.
2. Для `delta` = 0, 2, 4, 6, 8 отсчетов (8 отсчетов - половина символа) при $E_b/N_0 = 10$ дБ и $2\cdot10^4$ битов посчитать BER и EVM.
3. Повторить п. 2 для $\alpha = 0.1$ и $\alpha = 0.9$: передать одинаковое значение `alpha=...` в `shape_symbols` и в `apply_matched_filter` (как в задании 4). Построить на одном графике EVM(`delta`) для $\alpha$ = 0.1, 0.35 и 0.9.
4. Объяснить результат с помощью глазковых диаграмм из задания 4.

### Контрольные вопросы

1. Что такое синфазная и квадратурная составляющие? Почему по одной несущей можно передавать два независимых сигнала?
2. Сколько битов передает один символ QPSK? Как связаны $R_b$ и $R_s$, $E_b$ и $E_s$?
3. Что такое код Грея и зачем он нужен? Выполняется ли он для таблицы отображения из раздела 2?
4. Что такое межсимвольная интерференция? Сформулируйте критерий Найквиста.
5. Почему на передатчике и приемнике ставят фильтры RRC, а не один фильтр RC? Устраняет ли МСИ один RRC-фильтр?
6. Как коэффициент скругления $\alpha$ влияет на полосу сигнала и на глазковую диаграмму?
7. Зачем на приемнике нужен согласованный фильтр? Какую еще функцию он выполняет после квадратурного демодулятора?
8. Как связаны SNR и $E_b/N_0$? Почему системы связи сравнивают по $E_b/N_0$?
9. Почему помехоустойчивость QPSK и BPSK в пересчете на бит одинакова?
10. Как сдвиг фазы и сдвиг частоты несущей влияют на созвездие? Какие устройства приемника их компенсируют?